In [ ]:
import sys, os, warnings, subprocess
subprocess.run(["pip", "install", ".", "--quiet"], check=True,
               cwd="/mnt/storage_ssd_raid/research_projects/intrinsic_neural_timescales")
warnings.simplefilter("ignore", UserWarning)

import numpy as np
import scipy as sp
import pandas as pd

import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({
    'font.size':         8,
    'figure.dpi':        300,
    'svg.fonttype':      'none',
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Roboto'],
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

sys.path.extend([r'/mnt/storage_ssd_raid/research_projects/snaplab_tools'])

from src import nulls
from src.notebook import load_notebook_data, average_pet_markers
from src.timescales import create_ints_dataframe_custom_grouping, compute_deflections_glm
from src.plotting import plot_faceted_lollipops
from snaplab_tools.plotting.plotting import plot_correlation, plot_brain_surface_data
from snaplab_tools.plotting.utils import get_my_colors

In [ ]:
my_colors = get_my_colors(cat_trio=False, as_list=True)
brain_map  = 'SA_Axis'

# Load data

In [ ]:
dataset = 'HCP-YA'
which_modality = 'FMRI'
# meg_freq = '_lp-0.5hz'
use_7t = False
# output_type = 'timescale_zero_crossing'
# output_type = 'timescale_zero_crossing_robust_nc3'
output_type = 'timescale_auc_zero_crossing'

# output_type = output_type + '_lp-0.01hz'
# output_type = output_type + '_hp-0.01hz'
# output_type = output_type + '_bp-0.01-0.08hz'
# output_type = output_type + '_bp-0.08-0.15hz'

notebook_data = load_notebook_data(dataset=dataset, which_modality=which_modality,
                                   use_7t=use_7t, output_type=output_type,
                                   load_subject_data=False)

In [ ]:
left_hemi = False

In [ ]:
results_dir     = notebook_data['results_dir']
nas_scratch_dir = f'/mnt/lindenmp_nas/lindenmp/work/figure_scratch/intrinsic_neural_timescales/{dataset}'
os.makedirs(nas_scratch_dir, exist_ok=True)

In [ ]:
# get key outputs for this notebook
timescales_processor = notebook_data['timescales_processor']
data_dict            = notebook_data['data_dict']
data_dict_tsnr       = notebook_data['data_dict_tsnr']

df_neuro       = notebook_data['supporting_data']['df_neuro'].copy()
pet_categories = notebook_data['supporting_data']['pet_categories']
yeo_systems    = notebook_data['supporting_data']['yeo_systems']

if left_hemi:
    df_neuro = df_neuro.iloc[:200, :] # retain left hemi
    yeo_systems = yeo_systems[:200]

print(data_dict.keys())
print(df_neuro.shape, df_neuro.columns)

# Data for figures

In [ ]:
subject_ids = list(dict.fromkeys(scan.subject_id for scan in timescales_processor.scan_registry))
n_subs = len(subject_ids)
print(f'Number of subjects: {n_subs}')

## PET data

In [ ]:
pet_categories.head()

In [ ]:
pet_categories.to_csv(os.path.join(results_dir, 'pet_categories.csv'))

In [ ]:
pet_category = 'neurotransmitter_function'
# pet_category = 'monoamine_status'
# pet_category = 'receptor_mechanism'
# pet_category = 'gpcr_signaling'
# pet_category = 'custom_grouping'

plot_cols = pet_categories[pet_category].unique()
plot_cols = [f'Mean_{s}' for s in plot_cols]

if pet_category == 'neurotransmitter_function':
    order_indices = [1, 0, 2]
    plot_cols = [plot_cols[i] for i in order_indices]

if pet_category == 'receptor_mechanism':
    order_indices = [0, 2, 1]
    plot_cols = [plot_cols[i] for i in order_indices]

print(plot_cols)

In [ ]:
# hold onto SA axis
sa_axis = df_neuro['SA_Axis'].copy()

# PET only
neuro_cols = pet_categories.index
print(list(neuro_cols))
df_neuro = df_neuro[neuro_cols]
for col in neuro_cols:
    df_neuro[col] = sp.stats.zscore(df_neuro[col])

neuro_cols = [s.split('_')[1] for s in neuro_cols] # remove PET prefix
neuro_cols = ['_'.join(s) for s in zip(*[list(pet_categories[pet_category]), neuro_cols])] # append PET category

print(neuro_cols)
df_neuro.columns = neuro_cols
pet_categories.index = neuro_cols

In [ ]:
# df_neuro = df_neuro.rename(columns={"SA_Axis": "Hierarchy_SA-Axis", "SF_Axis": "Hierarchy_SF-Axis", "Micro_Axis": "Hierarchy_Micro-Axis"})
# df_neuro.drop(['Gene_PC1', 'Gene_PC2', 'Gene_PC3', 'PET_PC1', 'PET_PC2', 'PET_PC3'], axis=1, inplace=True)
df_tmp = average_pet_markers(df_neuro, pet_categories, pet_category)
df_neuro = pd.concat((df_neuro, df_tmp), axis=1)
df_neuro.to_csv(os.path.join(results_dir, 'df_neuro.csv'))
df_neuro

## INT data

In [ ]:
collapse_hcp = True
sort_by_median = True

if which_modality == 'FMRI' and 'HCP' in dataset:
    if collapse_hcp:
        df_ints = create_ints_dataframe_custom_grouping(data_dict, sort_by_median=sort_by_median,
                                                        average_all_rest=True, average_7t_movies=True, average_3t_tasks=True)
        df_tsnr = create_ints_dataframe_custom_grouping(data_dict_tsnr, sort_by_median=sort_by_median,
                                                        average_all_rest=True, average_7t_movies=True, average_3t_tasks=True)
    else:
        df_ints = create_ints_dataframe_custom_grouping(data_dict, sort_by_median=sort_by_median,
                                                        average_7t_rest=True, average_3t_rest=True, average_all_rest=False,
                                                        average_3t_task_pairs=True, average_7t_movies=True)
        df_tsnr = create_ints_dataframe_custom_grouping(data_dict_tsnr, sort_by_median=sort_by_median,
                                                        average_7t_rest=True, average_3t_rest=True, average_all_rest=False,
                                                        average_3t_task_pairs=True, average_7t_movies=True)
else:
    df_ints = create_ints_dataframe_custom_grouping(data_dict, sort_by_median=sort_by_median)
    df_tsnr = create_ints_dataframe_custom_grouping(data_dict_tsnr, sort_by_median=sort_by_median)

if collapse_hcp:
    reference = 'task-Rest'
    condition = 'task-Tasks'
else:
    reference = 'task-REST_3T'
    condition = 'task-WM'

# filter out low-SNR regions
q = 0.1
region_mask = np.logical_or(df_tsnr[reference] < np.quantile(df_tsnr[reference], q=q),
                             df_tsnr[condition] < np.quantile(df_tsnr[condition], q=q))
region_mask = region_mask.values
for col in df_ints.columns:
    df_ints.loc[:, col][region_mask] = np.nan

if left_hemi:
    df_ints = df_ints.iloc[:200, :]

df_ints

## BrainSMASH spatial nulls

In [ ]:
# Spatial nulls now come from src.nulls: per-hemisphere GEODESIC BrainSMASH surrogates, the single
# source of truth shared across the figure notebooks (replaces the old Euclidean, whole-brain
# nulls). Surrogates are generated inside correlate_family below and cached under
# results/<dataset>/brainsmash_nulls, so BrainSMASH only runs once per map.
n_perms   = 5000
n_regions = df_neuro.shape[0]

# Per-hemisphere geodesic distance basis + hemisphere labels + the on-disk surrogate cache dir.
distance_matrix, hemi_labels = nulls.load_distance_matrix(n_regions, kind='geodesic')
surrogate_cache_dir = nulls.surrogate_cache_dir(timescales_processor, dataset)

# Figure 3: brain plots of PET maps

In [ ]:
if not left_hemi:
    for col in plot_cols:
    # for col in df_neuro.columns:
        out_file = os.path.join(nas_scratch_dir, f'{col}.png')

        if 'PCA' in col:
            data_vector = df_neuro[col].values
            cmap = 'coolwarm'
            symmetric_cbar = True
            cbar_label = 'PC Score'
        else:
            data_vector = sp.stats.rankdata(df_neuro[col], nan_policy='omit').astype(int)
            cmap = 'magma'
            symmetric_cbar = False
            cbar_label = 'Expression (rank)'

        # if not os.path.exists(out_file):
        fig, axes = plot_brain_surface_data(
            data_vector=data_vector,
            parcellation='schaefer_400-7',
            # title=col.split('_')[1],
            title=None,
            cmap=cmap, symmetric_cbar=symmetric_cbar, cbar_label=cbar_label
        )
        plt.show()
        fig.savefig(os.path.join(nas_scratch_dir, f'{col}.png'), bbox_inches='tight', dpi=1000, transparent=True)
            
        # fig, axes = plt.subplots(nrows=1, ncols=1, sharex=False, sharey=False, figsize=(2.25, 1.75))
        # plot_correlation(sa_axis, data_vector, axes,
        #                  y_label=col,
        #                  data_group=np.asarray(yeo_systems), show_stats=True, fontsize=8, method='spearman',
        #                  size=15,
        #                  cbar_label='Yeo Systems',
        #                  auto_polynomial=True, models_to_test=[1,], model_selection_metric='variance_explained')
        # plt.show()

# Figure 3: Deflection correlation, scatter

In [ ]:
df_neuro.head()

In [ ]:
results = compute_deflections_glm(df_ints.loc[:,reference], df_ints.loc[:,condition], df_neuro, standardize=True)

In [ ]:
# Partial correlation of task INT vs each PET map, controlling for rest INT, tested against a
# per-hemisphere geodesic BrainSMASH null. correlate_family surrogates each PET map (cached),
# residualizes both task and the surrogate on rest INT, and correlates — reproducing the observed
# partial r from compute_deflections_glm. p is one-tailed in the observed direction
# (two_tailed=False), matching this notebook's original directional test.
rest_vals = df_ints[reference].values   # covariate: rest INT
task_vals = df_ints[condition].values   # brain map: task INT

df_family = nulls.correlate_family(
    task_vals, df_neuro, covariates=rest_vals,
    n_regions=n_regions, n_perms=n_perms, two_tailed=False,
    cache_dir=surrogate_cache_dir, distance_matrix=distance_matrix, hemi=hemi_labels,
    desc='PET null corrs')

# One-tailed spatial p-value per map (name -> p), consumed by the scatter/lollipop cells below.
null_pvals = df_family['p_smash'].to_dict()

print('Example null p-values:', {k: f'{v:.3f}' for k, v in list(null_pvals.items())[:4]})

In [ ]:
results[col].keys()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=len(plot_cols),
                         sharex=False, sharey=False,
                         figsize=(len(plot_cols)*1.5, 1.75), dpi=300)

for i, col in enumerate(plot_cols):
    x_glm = pd.Series(results[col]['feature_partial_resid'], name=col)
    y_glm = pd.Series(results[col]['task_partial_resid'], name='INT Deflections')
    plot_correlation(x_glm, y_glm, axes[i],
                     x_label=col.split('_')[1],
                     data_group=np.asarray(yeo_systems),
                     show_stats=True, fontsize=8, method='pearson',
                     size=8, cbar_label='Yeo Systems', show_colorbar=False, data_group_cmap='yeo7',
                     auto_polynomial=False, models_to_test=[1,], model_selection_metric='variance_explained',
                     custom_pvalue=null_pvals[col])
    if i != 0:
        axes[i].set_ylabel('')

for ax in axes:
    ax.tick_params(axis='y', which='major', pad=1)
    ax.tick_params(axis='x', which='major', pad=1)

fig.tight_layout()
fig.savefig(os.path.join(nas_scratch_dir, f'corr_intdelta_neurobio.svg'), bbox_inches='tight', dpi=300, transparent=True)
plt.show()


In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=1,
                         sharex=False, sharey=False,
                         figsize=(2.5, 1.75), dpi=300)

plot_correlation(x_glm, y_glm, axes,
                    x_label=col.split('_')[1],
                    data_group=np.asarray(yeo_systems),
                    show_stats=True, fontsize=8, method='pearson',
                    size=8, cbar_label='Yeo Systems', show_colorbar=True, data_group_cmap='yeo7',
                    auto_polynomial=False, models_to_test=[1,], model_selection_metric='variance_explained')
    
fig.tight_layout()
fig.savefig(os.path.join(nas_scratch_dir, f'corr_intdelta_neurobio_cbar.svg'), bbox_inches='tight', dpi=300, transparent=True)
plt.show()

In [ ]:
# Lollipop inputs: observed partial r and its one-tailed geodesic BrainSMASH p-value (both from
# correlate_family above), restricted to the 19 individual PET markers. P-values are the raw
# spatial-permutation p (no FDR), as in the original figure.
df_results = df_family.loc[pet_categories.index, ['r']].rename(columns={'r': 'INT Deflections'})
df_pvals   = df_family.loc[pet_categories.index, ['p_smash']].rename(columns={'p_smash': 'INT Deflections'})

fig, axes = plot_faceted_lollipops(df_results, df_pvals, category_colors=my_colors,
                                   figsize=(1.75, 3.25), markersize=3, fontsize=8,
                                   show_legend=True, annotate_sig=True, nonsig_alpha=0.25)
df_results.to_csv(os.path.join(results_dir, 'df_results_delta.csv'))
df_pvals.to_csv(os.path.join(results_dir, 'df_pvals_delta.csv'))
fig.savefig(os.path.join(nas_scratch_dir, 'corr_intdelta_neurobio_lollipop.svg'), bbox_inches='tight', dpi=300, transparent=True)
plt.show()